In [23]:
!pip install annoy faiss-cpu hnswlib pandas numpy tabulate

In [24]:
import time
import numpy as np
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

def generate_data(num_samples, dim):
    """Membuat data vektor acak bertipe float32."""
    np.random.seed(42)
    return np.random.random((num_samples, dim)).astype('float32')

def benchmark_annoy(data, queries, metric, dim, n_trees=10, k=10):
    annoy_metric = 'angular' if metric == 'cosine' else 'euclidean'
    index = AnnoyIndex(dim, annoy_metric)

    # 1. Build Time
    start_time = time.time()
    for i, vec in enumerate(data):
        index.add_item(i, vec)
    index.build(n_trees)
    build_time = time.time() - start_time

    # 2. Query Time
    start_time = time.time()
    for q in queries:
        index.get_nns_by_vector(q, k)
    query_time_ms = ((time.time() - start_time) / len(queries)) * 1000

    return build_time, query_time_ms

def benchmark_faiss(data, queries, metric, dim, k=10):
    data_faiss = data.copy()
    queries_faiss = queries.copy()

    if metric == 'cosine':
        faiss.normalize_L2(data_faiss)
        faiss.normalize_L2(queries_faiss)
        index = faiss.IndexFlatIP(dim)  # Inner Product untuk Cosine
    else:
        index = faiss.IndexFlatL2(dim)  # L2 untuk Euclidean

    # 1. Build Time
    start_time = time.time()
    index.add(data_faiss)
    build_time = time.time() - start_time

    # 2. Query Time
    start_time = time.time()
    index.search(queries_faiss, k)
    query_time_ms = ((time.time() - start_time) / len(queries_faiss)) * 1000

    return build_time, query_time_ms

def benchmark_hnsw(data, queries, metric, dim, k=10):
    space = 'cosine' if metric == 'cosine' else 'l2'
    index = hnswlib.Index(space=space, dim=dim)

    # 1. Build Time
    start_time = time.time()
    index.init_index(max_elements=len(data), ef_construction=200, M=16)
    index.add_items(data)
    build_time = time.time() - start_time

    # 2. Query Time
    index.set_ef(50)
    start_time = time.time()
    index.knn_query(queries, k=k)
    query_time_ms = ((time.time() - start_time) / len(queries)) * 1000

    return build_time, query_time_ms

# Skenario pengujian sesuai instruksi
scenarios = [
    {"samples": 1000, "dim": 2, "metric": "euclidean"},
    {"samples": 1000, "dim": 2, "metric": "cosine"},
    {"samples": 1000000, "dim": 5, "metric": "euclidean"},
    {"samples": 1000000, "dim": 5, "metric": "cosine"}
]

results = []

for sc in scenarios:
    n, d, m = sc["samples"], sc["dim"], sc["metric"]
    print(f"Running benchmark for {n} samples, {d}D, metric: {m}...")

    data = generate_data(n, d)
    queries = generate_data(100, d) # 100 query vectors

    # Run Annoy
    t_build_annoy, t_query_annoy = benchmark_annoy(data, queries, m, d)

    # Run Faiss
    t_build_faiss, t_query_faiss = benchmark_faiss(data, queries, m, d)

    # Run HNSW
    t_build_hnsw, t_query_hnsw = benchmark_hnsw(data, queries, m, d)

    label = f"{n}/{d}D ({m})"
    results.append({
        "Jumlah Data/Dimensi (Metric)": label,
        "ANNOY (Build / Query)": f"{t_build_annoy:.4f}s / {t_query_annoy:.4f}ms",
        "FAISS (Build / Query)": f"{t_build_faiss:.4f}s / {t_query_faiss:.4f}ms",
        "HNSW (Build / Query)": f"{t_build_hnsw:.4f}s / {t_query_hnsw:.4f}ms",
    })

df_results = pd.DataFrame(results)
print("\n" + df_results.to_markdown(index=False))

Running benchmark for 1000 samples, 2D, metric: euclidean...
Running benchmark for 1000 samples, 2D, metric: cosine...
Running benchmark for 1000000 samples, 5D, metric: euclidean...
Running benchmark for 1000000 samples, 5D, metric: cosine...

| Jumlah Data/Dimensi (Metric)   | ANNOY (Build / Query)   | FAISS (Build / Query)   | HNSW (Build / Query)   |
|:-------------------------------|:------------------------|:------------------------|:-----------------------|
| 1000/2D (euclidean)            | 0.0565s / 0.0338ms      | 0.0001s / 0.0771ms      | 0.1164s / 0.0208ms     |
| 1000/2D (cosine)               | 0.0697s / 0.0703ms      | 0.0000s / 0.0789ms      | 0.1157s / 0.0172ms     |
| 1000000/5D (euclidean)         | 22.4877s / 0.0867ms     | 0.0180s / 7.8582ms      | 159.5522s / 0.0413ms   |
| 1000000/5D (cosine)            | 27.4343s / 0.0576ms     | 0.0148s / 6.9955ms      | 155.5887s / 0.0372ms   |
